#JSON

Json is a semi structured format. It helps in optimizing space as compared to tables.
we usually get this data when we hit api eg recall Airflow api hit

##Types of JSON

there are 2 types of JSON

1) line delimited (default)
2) Multiline

Multiline Json is slower since in multiline json the pyspark reads the whole data as one object and then segreagates each json individually. However in line delimited it considers each line as a single complete json. In case if finds a break in line delimited it assumes the files ended there and does not process further.



###sample JSON data
---


line_delimited_json

{"name":"Manish","age":20,"salary":20000},
{"name":"Nikita","age":25,"salary":21000},
{"name":"Pritam","age":16,"salary":22000},
{"name":"Prantosh","age":35,"salary":25000},
{"name":"Vikash","age":67,"salary":40000}

---

single_file_json with extra fields

{"name":"Manish","age":20,"salary":20000},
{"name":"Nikita","age":25,"salary":21000},
{"name":"Pritam","age":16,"salary":22000},
{"name":"Prantosh","age":35,"salary":25000},
{"name":"Vikash","age":67,"salary":40000,"gender":"M"}

---

corrupted_json

{"name":"Manish","age":20,"salary":20000},
{"name":"Nikita","age":25,"salary":21000},
{"name":"Pritam","age":16,"salary":22000},
{"name":"Prantosh","age":35,"salary":25000},
{"name":"Vikash","age":67,"salary":40000

---

Multi_line_incorrect

{
  "name": "Manish",
  "age": 20,
  "salary": 20000
},
{
  "name": "Nikita",
  "age": 25,
  "salary": 21000
},
{
  "name": "Pritam",
  "age": 16,
  "salary": 22000
},
{
  "name": "Prantosh",
  "age": 35,
  "salary": 25000
},
{
  "name": "Vikash",
  "age": 67,
  "salary": 40000
}

Multi_line_correct
[
{
  "name": "Manish",
  "age": 20,
  "salary": 20000
},
{
  "name": "Nikita",
  "age": 25,
  "salary": 21000
},
{
  "name": "Pritam",
  "age": 16,
  "salary": 22000
},
{
  "name": "Prantosh",
  "age": 35,
  "salary": 25000
},
{
  "name": "Vikash",
  "age": 67,
  "salary": 40000
}
]


In [0]:
'''
{"name":"Manish","age":20,"salary":20000},
{"name":"Nikita","age":25,"salary":21000},
{"name":"Pritam","age":16,"salary":22000},
{"name":"Prantosh","age":35,"salary":25000},
{"name":"Vikash","age":67,"salary":40000,"gender":"M"}  ##tuple with extra column
'''


spark.read.format("json")\
  .option("mode","PERMISSIVE")\
  .option("inferSchema","true")\
  .load("/Volumes/workspace/default/workspace_volume/extraFieldJson.json")\
  .show()

### why the extra null row?

Spark’s JSON reader expects either:
One JSON object per line (JSON Lines / NDJSON)

{"name":"Manish","age":20,"salary":20000}
{"name":"Nikita","age":25,"salary":21000}

✅ works perfectly

A proper JSON array

[
  {"name":"Manish","age":20,"salary":20000},
  {"name":"Nikita","age":25,"salary":21000}
]

✅ also works

Your file is effectively seen as one continuous stream:

{"name":"Manish",...},{"name":"Nikita",...},{"name":"Pritam",...}...

Spark reads your file as a stream of stuff, not as lines. It walks through character by character looking for valid JSON objects ({...}).

Those commas don’t just “break” every line. Instead:
Spark’s parser doesn’t necessarily emit an error per line—it depends on how it chunks the input internally.

In your case:

Most commas get “absorbed” during parsing of adjacent objects
But at least one comma is interpreted as a separate invalid record
That results in exactly one corrupt row → one NULL row

Bottom line
It’s not “one bad line → one NULL row”
It’s “one invalid JSON fragment → one NULL row”
In your file, at least one comma is parsed as its own fragment

###why the extra column?

When one json has 4 entries and all others have only 3 entries then the table will be created for 4 entries and all the entires with 3 values will be appended with a null

#Reading Multiline Json

### Without multiLine option

In [0]:
spark.read.format("json") \
  .option("mode", "PERMISSIVE") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/workspace_volume/Multiline.json") \
  .show(truncate=False)

### with multiLine option

In [0]:
spark.read.format("json") \
  .option("mode", "PERMISSIVE") \
  .option("multiLine", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/workspace_volume/Multiline.json") \
  .show(truncate=False)

##INVALID MULTILINE JSON

when json is not sent as an list of dictionary

In [0]:
spark.read.format("json") \
  .option("mode", "PERMISSIVE") \
  .option("multiLine", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/workspace_volume/InvalidMultiline.json") \
  .show(truncate=False)

##note that it only reads one value and one null row since after first row it just assumes that these are the only records because list of dictionary was not given

#Corrupted JSON

In [0]:
spark.read.format("json") \
  .option("mode", "PERMISSIVE") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/workspace_volume/CorruptedJson.json") \
  .show(truncate=False)

### again note that this only worked because the json was not multiline. had it been multiline the corrupted records would not be shown check below

In [0]:
spark.read.format("json") \
  .option("mode", "PERMISSIVE") \
  .option("inferSchema", "true") \
  .option("multiline","true")\
  .load("/Volumes/workspace/default/workspace_volume/CorruptedJson.json") \
  .show(truncate=False)

How Spark reads:
Treats the entire file as ONE JSON document
Uses a whole-file parser, not line-by-line
What happens internally:

Your file effectively becomes:

{obj1},
{obj2},
{obj3},
{obj4},
{BROKEN obj5

This is:

Not a valid JSON array ❌
Not valid JSON overall ❌
Spark’s behavior in PERMISSIVE mode:
It tries to extract valid objects it can parse
When it hits the malformed tail (Vikash record), it can’t recover structurally
Instead of keeping raw corrupt text, it emits:
NULL NULL NULL

👉 No _corrupt_record column here because:

The parser isn’t working per-record anymore
It’s working on a single JSON structure
It loses the exact boundary of the bad object
⚡ Key Difference (this is the takeaway)
Mode	Parsing Strategy	Corrupt Handling
multiLine = false	line-by-line	keeps bad row in _corrupt_record
multiLine = true	whole file	emits NULL row (loses raw text)